In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
pip install transformers datasets accelerate peft bitsandbytes sentencepiece scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import torch
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from sklearn.metrics import accuracy_score

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "microsoft/phi-2"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Fix padding token issue
tokenizer.pad_token = tokenizer.eos_token

# 8-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.pad_token_id = tokenizer.pad_token_id

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 5,242,880 || all params: 2,784,926,720 || trainable%: 0.1883


In [6]:
def load_dataset_json(path):

    with open(path) as f:
        data = json.load(f)

    texts = []

    for ex in data:

        question = ex["question"]
        choices = ex["choices"]
        answer = ex["answer"]

        prompt = f"""Question: {question}

A. {choices[0]}
B. {choices[1]}
C. {choices[2]}
D. {choices[3]}

Answer:"""

        target = ["A","B","C","D"][answer]

        texts.append(prompt + " " + target)

    return Dataset.from_dict({"text": texts})

In [7]:
def tokenize(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )


In [8]:
def train_model(dataset, output_dir):

    dataset = dataset.map(tokenize)

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=20,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        fp16=True
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=data_collator
    )

    trainer.train()

    trainer.save_model(output_dir)

In [9]:
verbatim_dataset = load_dataset_json("/kaggle/input/datasets/debayushdey/contamination-finetuning/verbatim_data.json")

train_model(
    verbatim_dataset,
    output_dir="Model_V"
)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
10,1.869595
20,1.764726
30,1.679987
40,1.620661
50,1.593632
60,1.572098
70,1.525176
80,1.483229
90,1.441030
100,1.386626


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentra

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

paraphrased_dataset = load_dataset_json("/kaggle/input/datasets/debayushdey/contamination-finetuning/paraphrased_200.json")

train_model(
    paraphrased_dataset,
    output_dir="Model_P"
)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Step,Training Loss
10,1.787038
20,1.582725
30,1.568396
40,1.512687
50,1.505820
60,1.425750
70,1.444292
80,1.398919
90,1.286937
100,1.308630


In [11]:
def evaluate_accuracy(model, dataset_json):

    with open(dataset_json) as f:
        data = json.load(f)

    preds = []
    labels = []

    for ex in data:

        question = ex["question"]
        choices = ex["choices"]
        answer = ex["answer"]

        prompt = f"""Question: {question}

A. {choices[0]}
B. {choices[1]}
C. {choices[2]}
D. {choices[3]}

Answer:"""

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        output = model.generate(
            **inputs,
            max_new_tokens=2
        )

        text = tokenizer.decode(output[0], skip_special_tokens=True)

        predicted = text.split("Answer:")[-1].strip()[0]

        preds.append(predicted)

        labels.append(["A","B","C","D"][answer])

    acc = accuracy_score(labels, preds)

    return acc

In [12]:

import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [34]:
!nvidia-smi

Thu Mar  5 17:47:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             31W /   70W |    9223MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

BASE_MODEL = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model_v = PeftModel.from_pretrained(
    base_model,
    "Model_V"
)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [14]:
acc_v = evaluate_accuracy(
    model_v,
    "/kaggle/input/datasets/debayushdey/contamination-finetuning/verbatim_data.json"
)

print("Model_V Training Accuracy:", acc_v)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Model_V Training Accuracy: 0.985


In [15]:
del model_v
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [16]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model_p = PeftModel.from_pretrained(
    base_model,
    "Model_P"
)
acc_p = evaluate_accuracy(
    model_p,
    "/kaggle/input/datasets/debayushdey/contamination-finetuning/paraphrased_200.json"
)

print("Model_P Training Accuracy:", acc_p)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Model_P Training Accuracy: 0.985


In [17]:
del model_p
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [18]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model_v = PeftModel.from_pretrained(
    base_model,
    "Model_V"
)
acc_v = evaluate_accuracy(
    model_v,
    "/kaggle/input/datasets/debayushdey/contamination-finetuning/paraphrased_200.json"
)

print("Model_V Accuracy on paraphrase dataset:", acc_v)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Model_V Accuracy on paraphrase dataset: 0.525


In [20]:
del model_v
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

NameError: name 'model_v' is not defined

In [21]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model_p = PeftModel.from_pretrained(
    base_model,
    "Model_P"
)
acc_p = evaluate_accuracy(
    model_p,
    "/kaggle/input/datasets/debayushdey/contamination-finetuning/verbatim_data.json"
)

print("Model_P Accuracy on verbatim dataset:", acc_p)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Model_P Accuracy on verbatim dataset: 0.63


In [29]:
del model_p
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

NameError: name 'model_p' is not defined

In [23]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model_p = PeftModel.from_pretrained(
    base_model,
    "Model_P"
)
acc_p = evaluate_accuracy(
    model_p,
    "/kaggle/input/datasets/debayushdey/contamination-finetuning/clean_data.json"
)

print("Model_P Accuracy on clean dataset:", acc_p)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Model_P Accuracy on clean dataset: 0.505


In [25]:
del model_p
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

NameError: name 'model_p' is not defined

In [26]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model_v = PeftModel.from_pretrained(
    base_model,
    "Model_V"
)
acc_v = evaluate_accuracy(
    model_v,
    "/kaggle/input/datasets/debayushdey/contamination-finetuning/clean_data.json"
)

print("Model_V Accuracy on clean dataset:", acc_v)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 50.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 9.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.29 GiB is allocated by PyTorch, and 114.72 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [38]:
!ls Model_V

adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  tokenizer_config.json  training_args.bin


In [40]:
!zip -r Model_P.zip Model_P
!zip -r Model_V.zip Model_V

updating: Model_P/ (stored 0%)
updating: Model_P/training_args.bin (deflated 54%)
updating: Model_P/tokenizer.json (deflated 82%)
updating: Model_P/README.md (deflated 65%)
updating: Model_P/adapter_model.safetensors (deflated 8%)
updating: Model_P/adapter_config.json (deflated 57%)
updating: Model_P/tokenizer_config.json (deflated 49%)
updating: Model_V/ (stored 0%)
updating: Model_V/training_args.bin (deflated 54%)
updating: Model_V/tokenizer.json (deflated 82%)
updating: Model_V/README.md (deflated 65%)
updating: Model_V/adapter_model.safetensors (deflated 8%)
updating: Model_V/adapter_config.json (deflated 57%)
updating: Model_V/tokenizer_config.json (deflated 49%)
